In [ ]:
%load_ext autoreload
%autoreload 2

In [1]:
import crosscoders as xc


------------------------- CONSTANTS -------------------------
GlobalsConfig(
    PROJECT_ROOT_DIR = '/home/ec2-user/crosscoders',
    CONFIG_FILEPATH = '/home/ec2-user/crosscoders/src/scripts/configs/train.yml',
    EXPERIMENT = ExperimentConfig(
        BATCH_SIZE = 8192,
        MAX_RECORDS = None,
        MAX_BATCHES = 1000000,
        MAX_TOKENS = 1000,
        NUM_GPUS = 1,
        NUM_TRAINERS = 1,
        HARDWARE = HardwareConfig(
            dtype = torch.float32,
            device = 'cuda',
        ),
    ),
)
-------------------------------------------------------------



In [3]:
import torch
import matplotlib.pyplot as plt

In [4]:
from crosscoders.constants import CONSTANTS
from crosscoders.dataclasses.configs.runner import RunnerConfig
from crosscoders.utils import from_dict, get_config


runner_cfg = from_dict(
    RunnerConfig,
    get_config(CONSTANTS.CONFIG_FILEPATH).get('RUNNER', {})
)
runner_cfg

RunnerConfig(
    MODEL = ModelConfig(
        CAUSALITY = 'acausal',
        LOCALITY = 'global',
        N_LAYERS = 12,
        D_MODEL = 768,
        D_CODER = 16384,
        HARDWARE = HardwareConfig(
            dtype = torch.float32,
            device = 'cuda',
        ),
    ),
    LOSS = AcausalLossConfig(
        L1_COEFFICIENT = 1.0,
    ),
    OPTIMIZER = OptimizerConfig(
        optimizer = <class 'torch.optim.adam.Adam'>,
        parameters = OptimizerParameters(
            lr = 0.002279,
            betas = (0.9, 0.999),
            fused = True,
        ),
    ),
)

In [8]:
checkpoint_path = '/home/ec2-user/ray_results/TorchTrainer_2025-02-13_18-26-34/TorchTrainer_0db33_00000_0_2025-02-13_18-26-34/checkpoint_000000/model.pt'

In [9]:
from crosscoders.autoencoders.acausal.model import AcausalAutoencoder


model = AcausalAutoencoder(runner_cfg.MODEL)


# with checkpoint.as_directory() as checkpoint_dir
checkpoint_dict = torch.load(checkpoint_path)
model.load_state_dict(checkpoint_dict)

/tmp/ipykernel_22637/619853545.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint_dict = torch.load(checkpoint_path)


<All keys matched successfully>

In [18]:
model.to(runner_cfg.MODEL.HARDWARE.device)
model.eval()

AcausalAutoencoder()

In [10]:
from crosscoders.data.dataset import TinyStoriesRayDataset


train_ds = TinyStoriesRayDataset().load('activations')

train_dl = train_ds.iter_torch_batches(
    batch_size=CONSTANTS.EXPERIMENT.BATCH_SIZE,
    # local_shuffle_buffer_size=16
    device=CONSTANTS.EXPERIMENT.HARDWARE.device
)


Metadata Fetch Progress 0:   0%|          | 0.00/83.0 [00:00<?, ? task/s]

2025-02-13 19:11:07,146	INFO worker.py:1654 -- Connecting to existing Ray cluster at address: 10.0.1.126:6379...
2025-02-13 19:11:07,157	INFO worker.py:1832 -- Connected to Ray cluster. View the dashboard at http://127.0.0.1:8265 


Parquet Files Sample 0:   0%|          | 0.00/5.00 [00:00<?, ? file/s]

(ReadParquet->SplitBlocks(2) pid=30114) Traceback (most recent call last):
(ReadParquet->SplitBlocks(2) pid=30114)   File "pyarrow/public-api.pxi", line 145, in pyarrow.lib.pyarrow_wrap_data_type
(ReadParquet->SplitBlocks(2) pid=30114)   File "pyarrow/types.pxi", line 606, in pyarrow.lib.LargeListType.init
(ReadParquet->SplitBlocks(2) pid=30114)   File "pyarrow/types.pxi", line 232, in pyarrow.lib.DataType.init
(ReadParquet->SplitBlocks(2) pid=30114)   File "pyarrow/types.pxi", line 105, in pyarrow.lib._datatype_to_pep3118
(ReadParquet->SplitBlocks(2) pid=30114)   File "/home/ec2-user/crosscoders/.conda/lib/python3.12/site-packages/ray/air/util/tensor_extensions/arrow.py", line 461, in __arrow_ext_deserialize__
(ReadParquet->SplitBlocks(2) pid=30114)     @classmethod
(ReadParquet->SplitBlocks(2) pid=30114) 
(ReadParquet->SplitBlocks(2) pid=30114) KeyboardInterrupt: 
(ReadParquet->SplitBlocks(2) pid=30111) 
(ReadParquet->SplitBlocks(2) pid=30245) 
(ReadParquet->SplitBlocks(2) pid=30110)

In [11]:
batch = train_ds.take_batch()

2025-02-13 19:40:29,482	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/session_2025-02-13_17-56-48_770585_3642/logs/ray-data
2025-02-13 19:40:29,483	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> LimitOperator[limit=1000] -> LimitOperator[limit=20]


Running 0: 0.00 row [00:00, ? row/s]

- ReadParquet->SplitBlocks(2) 1: 0.00 row [00:00, ? row/s]

- limit=1000 2: 0.00 row [00:00, ? row/s]

- limit=20 3: 0.00 row [00:00, ? row/s]

2025-02-13 19:40:34,556	ERROR worker.py:422 -- Unhandled error (suppress with 'RAY_IGNORE_UNHANDLED_ERRORS=1'): ray::ReadParquet->SplitBlocks(2)() (pid=30114, ip=10.0.1.126)
    for b_out in map_transformer.apply_transform(iter(blocks), ctx):
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ec2-user/crosscoders/.conda/lib/python3.12/site-packages/ray/data/_internal/execution/operators/map_transformer.py", line 451, in __call__
    for block in blocks:
                 ^^^^^^
  File "/home/ec2-user/crosscoders/.conda/lib/python3.12/site-packages/ray/data/_internal/execution/operators/map_transformer.py", line 392, in __call__
    for data in iter:
                ^^^^
  File "/home/ec2-user/crosscoders/.conda/lib/python3.12/site-packages/ray/data/_internal/execution/operators/map_transformer.py", line 253, in __call__
    yield from self._block_fn(input, ctx)
  File "/home/ec2-user/crosscoders/.conda/lib/python3.12/site-packages/ray/data/_internal/planne

In [21]:
x = batch['resid_post']
x = torch.as_tensor(x, device=runner_cfg.MODEL.HARDWARE.device)
x.shape

torch.Size([20, 12, 768])

In [22]:
x_hat = model(x)
x_hat.shape

torch.Size([20, 12, 768])

In [24]:
model.W_dec.shape

torch.Size([16384, 12, 768])